In [ ]:
!pip install llama-index llama-index-llms-openai graspologic pyvis pandas nest-asyncio python-dotenv

### Explanation

This code is setting up a **Knowledge Graph and GraphRAG system** using **LlamaIndex**. Libraries like `pandas` help manage data, `networkx` helps create and analyze graphs, and `dotenv` loads API keys securely. LlamaIndex is used as the main framework because it connects the documents, AI model, embeddings, knowledge graph, and query system together. `Document` is used to handle the input data, while `PropertyGraphIndex`, `EntityNode`, and `Relation` help create a graph of entities and their relationships. `GoogleGenAI` connects the system with the Gemini AI model, and `HuggingFaceEmbedding` converts text into vectors for semantic search. The `hierarchical_leiden` function helps group related nodes into communities. Overall, **LlamaIndex acts as the central framework that manages the process of turning documents into a knowledge graph and using that graph with an AI model to answer questions.**

In [3]:
import asyncio
import nest_asyncio
import pandas as pd
import networkx as nx
import os
from dotenv import load_dotenv

from llama_index.core import Document, PropertyGraphIndex, Settings
from llama_index.core.graph_stores.types import (
    EntityNode, KG_NODES_KEY, KG_RELATIONS_KEY, Relation
)
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.schema import TransformComponent, BaseNode
from llama_index.core.async_utils import run_jobs
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from graspologic.partition import hierarchical_leiden

nest_asyncio.apply()
load_dotenv()

print("✅ Imports ready")

✅ Imports ready


### LLM and Model Configuration

This code configures the **Gemini LLM, embedding model, and pipeline settings**. It loads multiple Gemini API keys from the `.env` file and uses `itertools.cycle()` to rotate between them. The `get_next_llm()` function creates a Gemini model using the next available API key, helping distribute API usage. Two LLM instances are created: `EXTRACTION_LLM` for extracting entities and relationships, and `QUERY_LLM` for answering user queries.

The code also uses the `BAAI/bge-small-en-v1.5` Hugging Face embedding model to convert text into vectors for semantic search. Finally, parameters such as the maximum number of articles, graph paths, workers, and cluster size are configured. `Settings.llm` and `Settings.embed_model` set the default LLM and embedding model for LlamaIndex.

In [11]:
import warnings
import itertools
import os
load_dotenv()
# — LLM Configuration ————————————————————
GEMINI_KEYS = [k for k in [
    os.getenv("GEMINI_API_KEY"),
    os.getenv("GEMINI_API_KEY_2"),
    os.getenv("GEMINI_API_KEY_3"),
] if k]  # filters out any keys that aren't set

if not GEMINI_KEYS:
    raise ValueError("No Gemini API keys found — check your .env file")

print(f"✅ {len(GEMINI_KEYS)} Gemini API key(s) loaded")
key_cycle = itertools.cycle(GEMINI_KEYS)

def get_next_llm():
    """Returns a Gemini LLM instance with the next key in rotation."""
    key = next(key_cycle)
    os.environ["GEMINI_API_KEY"] = key
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        from llama_index.llms.gemini import Gemini
        return Gemini(model="models/gemini-3.6-flash")

# Default LLMs (using first key)
EXTRACTION_LLM = get_next_llm()
QUERY_LLM      = get_next_llm()

# — Embedding Model ————————————————————
EMBED_MODEL = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# — Pipeline Parameters ————————————————————
MAX_ARTICLES         = 10
MAX_PATHS_PER_CHUNK  = 15
NUM_WORKERS          = 1
MAX_CLUSTER_SIZE     = 10
GRAPH_OUTPUT_FILE    = "ai_copyright_graph.html"

Settings.llm         = EXTRACTION_LLM
Settings.embed_model = EMBED_MODEL

print(f"✅ Using {EXTRACTION_LLM.model} for extraction, {QUERY_LLM.model} for querying")
print(f"✅ Using {EMBED_MODEL.model_name} for embeddings")


✅ 3 Gemini API key(s) loaded
✅ Using models/gemini-3.6-flash for extraction, models/gemini-3.6-flash for querying
✅ Using BAAI/bge-small-en-v1.5 for embeddings


### Entity and Relationship Types

This code defines the **types of entities and relationships** that the LLM should look for when reading the data. These act as **guidelines** for the LLM to create a structured knowledge graph. For example, the LLM can identify **OpenAI** as an `ORGANIZATION`, **GPT-4** as an `AI_SYSTEM`, and connect them using one of the defined relationship types. This helps keep the extracted data organized and consistent.

In [12]:
# — Entity types ————————————————————
# We'll embed these directly into the extraction prompt below.

ENTITY_TYPES = [
    "ORGANIZATION",  # Companies, labs, industry groups (OpenAI, Google, RIAA)
    "PERSON",        # Executives, policymakers, judges (Sam Altman, Thierry Breton)
    "LEGISLATION",   # Laws, acts, regulations (EU AI Act, DMCA, Copyright Act)
    "LEGAL_CASE",    # Lawsuits, court rulings (NYT v. OpenAI, Getty v. Stability AI)
    "CONCEPT",       # Abstract ideas: fair use, training data, IP rights
    "GOVERNMENT",    # Nations, regulatory bodies, courts (EU, US Copyright Office)
    "AI_SYSTEM",     # Specific models or products (GPT-4, Stable Diffusion, Gemini)
]

# — Relationship types ————————————————————
RELATION_TYPES = [
    "FILED_AGAINST",  # Plaintiff Organization/Person → LEGAL_CASE
    "DEFENDANT_IN",   # Organization/Person → LEGAL_CASE
    "REGULATES",      # GOVERNMENT/LEGISLATION → ORGANIZATION/AI_SYSTEM
    "ADVOCATES_FOR",  # ORGANIZATION/PERSON → CONCEPT or policy position
    "TRAINED_ON",     # AI_SYSTEM → CONCEPT or dataset type
    "PART_OF",        # PERSON → ORGANIZATION
    "REFERENCES",     # LEGAL_CASE/LEGISLATION → CONCEPT
    "OPPOSES",         # ORGANIZATION/GOVERNMENT → LEGISLATION/CONCEPT
]

print("✅ Ontology:")
print(f"   Entity types:       {ENTITY_TYPES}")
print(f"   Relationship types: {RELATION_TYPES}")

✅ Ontology:
   Entity types:       ['ORGANIZATION', 'PERSON', 'LEGISLATION', 'LEGAL_CASE', 'CONCEPT', 'GOVERNMENT', 'AI_SYSTEM']
   Relationship types: ['FILED_AGAINST', 'DEFENDANT_IN', 'REGULATES', 'ADVOCATES_FOR', 'TRAINED_ON', 'PART_OF', 'REFERENCES', 'OPPOSES']


### Knowledge Graph Extraction Prompt

This code creates a **prompt template** that will be given to the LLM to extract information from a news article. It dynamically adds the allowed `ENTITY_TYPES` and `RELATION_TYPES` that were defined earlier. The prompt tells the LLM to identify entities, such as organizations or AI systems, and find the relationships between them.

The `{{text}}` placeholder will later be replaced with the actual article, while `{{max_knowledge_triplets}}` sets the maximum number of relationships to extract. In simple terms, this prompt acts as **instructions for the LLM to convert an article into structured entities and relationships for the knowledge graph**.

In [13]:
# Build the entity and relationship type strings dynamically from our ontology
entity_types_str   = ", ".join(ENTITY_TYPES)
relation_types_str = ", ".join(RELATION_TYPES)

KG_TRIPLET_EXTRACT_TMPL = f"""
-Goal-
Given a news article about AI copyright, governance, or intellectual property,
identify all entities mentioned in the article and their relationships.

Extract up to {{max_knowledge_triplets}} entity-relation triplets.

-Allowed Entity Types-
{entity_types_str}

-Allowed Relationship Types-
{relation_types_str}

-Steps-
1. Identify ALL entities. For each entity extract:
   - name: Name of the entity, capitalized
   - type: One of the allowed entity types above
   - description: A brief description of the entity and its role in AI copyright/governance

2. Identify relationships between entities. For each pair extract:
   - source: name of the source entity
   - target: name of the target entity
   - relation: one of the allowed relationship types above
   - description: a sentence explaining why and how these entities are related

-Real Data-
######################
text: {{text}}
######################
"""

print("✅ Extraction prompt ready")
print(f"\nPreview (first 300 chars):\n{KG_TRIPLET_EXTRACT_TMPL[:300]}...")

✅ Extraction prompt ready

Preview (first 300 chars):

-Goal-
Given a news article about AI copyright, governance, or intellectual property,
identify all entities mentioned in the article and their relationships.

Extract up to {max_knowledge_triplets} entity-relation triplets.

-Allowed Entity Types-
ORGANIZATION, PERSON, LEGISLATION, LEGAL_CASE, CONC...


### Pydantic Models

This code defines a **fixed structure** for the LLM's output. It ensures that extracted entities and relationships follow the correct format and only use the allowed types before being added to the knowledge graph.

In [14]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal, List

EntityTypeStr = Literal[
    "ORGANIZATION", "PERSON", "LEGISLATION", "LEGAL_CASE",
    "CONCEPT", "GOVERNMENT", "AI_SYSTEM"
]
RelationTypeStr = Literal[
    "FILED_AGAINST", "DEFENDANT_IN", "REGULATES", "ADVOCATES_FOR",
    "TRAINED_ON", "PART_OF", "REFERENCES", "OPPOSES"
]

class ExtractedEntity(BaseModel):
    name: str = Field(description="Name of the entity, capitalized")
    type: EntityTypeStr = Field(description="One of the allowed entity types")
    description: str = Field(description="Brief description of the entity and its role")

class ExtractedRelationship(BaseModel):
    source: str = Field(description="Name of the source entity")
    target: str = Field(description="Name of the target entity")
    relation: RelationTypeStr = Field(description="One of the allowed relationship types")
    description: str = Field(description="Sentence explaining the relationship")

class ExtractionResult(BaseModel):
    entities: List[ExtractedEntity] = Field(default_factory=list)
    relationships: List[ExtractedRelationship] = Field(default_factory=list)

print("✅ Pydantic extraction models defined")

✅ Pydantic extraction models defined


### GraphRAG Extractor

This code creates a **custom extractor** that processes each text chunk using an LLM. It asks the LLM to identify entities and relationships and return them in JSON format. The output is validated using the Pydantic models, then converted into `EntityNode` and `Relation` objects and stored in the node's metadata.

It also includes **retry logic, API key rotation, JSON error handling, and asynchronous processing** to make extraction more reliable and efficient.

In [15]:
import json

class GraphRAGExtractor(TransformComponent):
    """
    Extracts entities and relationships WITH descriptions from each text chunk.
    Uses manual JSON prompting + parsing (not function/tool calling) for
    reliability across all LLM providers.
    """

    llm: LLM = Field(default_factory=lambda: Settings.llm)
    extract_prompt: PromptTemplate = Field(default_factory=lambda: PromptTemplate(KG_TRIPLET_EXTRACT_TMPL))
    num_workers: int = 4
    max_paths_per_chunk: int = 10

    @field_validator("extract_prompt", mode="before")
    @classmethod
    def coerce_to_prompt_template(cls, v):
        return PromptTemplate(v) if isinstance(v, str) else v

    def __call__(self, nodes, show_progress=False, **kwargs):
        return asyncio.run(self.acall(nodes, show_progress=show_progress, **kwargs))

    async def _aextract(self, node: BaseNode) -> BaseNode:
        text = node.get_content(metadata_mode="llm")

        base_prompt = self.extract_prompt.format(
            text=text, max_knowledge_triplets=self.max_paths_per_chunk
        )
        json_instructions = """

Respond with ONLY valid JSON in exactly this format, no extra text, no markdown fences:
{
  "entities": [
    {"name": "...", "type": "ORGANIZATION", "description": "..."}
  ],
  "relationships": [
    {"source": "...", "target": "...", "relation": "FILED_AGAINST", "description": "..."}
  ]
}"""
        full_prompt = base_prompt + json_instructions

        entities, relationships = [], []
        max_retries = 3
        for attempt in range(max_retries):
            try:
                await asyncio.sleep(4)
                # rotate key on each attempt
                current_llm = get_next_llm()
                response = await current_llm.acomplete(full_prompt)
                raw = response.text.strip()

                if not raw:
                    print(f"  Empty response (attempt {attempt+1})")
                    continue

                # Strip markdown code fences if present
                if raw.startswith("```"):
                    raw = raw.strip("`")
                    if raw.startswith("json"):
                        raw = raw[4:]
                    if "\n" in raw:
                        raw = raw.split("\n", 1)[1] if raw.startswith("\n") else raw
                    raw = raw.rsplit("```", 1)[0].strip()

                parsed = json.loads(raw)
                result = ExtractionResult.model_validate(parsed)
                entities      = result.entities
                relationships = result.relationships
                print(f"  ✅ Extracted {len(entities)} entities, {len(relationships)} relationships")
                break

            except json.JSONDecodeError as e:
                print(f"  JSON parse error (attempt {attempt+1}): {e}")
            except Exception as e:
                print(f"  Extraction error (attempt {attempt+1}): {e}")
                await asyncio.sleep(5)  # brief wait before rotating to next key

        existing_nodes     = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        base_metadata       = node.metadata.copy()

        existing_nodes += [
            EntityNode(
                name=entity.name,
                label=entity.type,
                properties={**base_metadata, "entity_description": entity.description},
            )
            for entity in entities
        ]

        entity_lookup = {e.name: e.type for e in entities}

        for rel in relationships:
            source_node = EntityNode(
                name=rel.source,
                label=entity_lookup.get(rel.source, "ENTITY"),
                properties=base_metadata,
            )
            target_node = EntityNode(
                name=rel.target,
                label=entity_lookup.get(rel.target, "ENTITY"),
                properties=base_metadata,
            )
            if rel.source not in entity_lookup:
                existing_nodes.append(source_node)
            if rel.target not in entity_lookup:
                existing_nodes.append(target_node)
            existing_relations.append(Relation(
                label=rel.relation,
                source_id=source_node.id,
                target_id=target_node.id,
                properties={**base_metadata, "relationship_description": rel.description},
            ))

        node.metadata[KG_NODES_KEY]     = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        return node

    async def acall(self, nodes, show_progress=False, **kwargs):
        jobs = [self._aextract(node) for node in nodes]
        return await run_jobs(
            jobs,
            workers=self.num_workers,
            show_progress=show_progress,
            desc="Extracting triplets",
        )

print("✅ GraphRAGExtractor defined (manual JSON mode)")

✅ GraphRAGExtractor defined (manual JSON mode)


### GraphRAG Store and Community Detection

This code creates a custom `GraphRAGStore` by extending LlamaIndex's `SimplePropertyGraphStore`. Its main purpose is to **find groups of closely related entities in the knowledge graph and generate a summary for each group**.

First, `build_communities()` converts the knowledge graph into a NetworkX graph and uses the **Hierarchical Leiden algorithm** to detect communities or clusters. For example, entities related to **OpenAI, copyright lawsuits, and AI training data** may be grouped into the same community.

The `_collect_community_info()` function then collects the **entities, their descriptions, and the relationships between them** for each community. Finally, `_generate_summaries()` sends this information to the LLM, which creates a short 3–5 sentence briefing explaining the main topics, connections, legal issues, and disputes in that community.

In simple terms:

```text
Knowledge Graph
       ↓
Find related groups using Leiden
       ↓
Collect entities + relationships
       ↓
Send each group to the LLM
       ↓
Generate a summary for each community

### Leiden Algorithm

Leiden is an algorithm that **finds groups of closely connected nodes in a graph**. In GraphRAG, it groups related entities together so the LLM can create a summary for each group.

In [16]:
class GraphRAGStore(SimplePropertyGraphStore):
    """
    Extends SimplePropertyGraphStore with:
    - Leiden community detection
    - LLM-generated community summaries (using entity + relationship descriptions)
    """

    community_summaries: dict = {}

    def build_communities(self):
        print("Running community detection...")
        nx_graph = self._to_networkx()

        if not nx_graph.nodes:
            print("⚠️  Graph is empty — no communities to detect")
            return

        print(f"Graph has {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")

        clusters = hierarchical_leiden(nx_graph, max_cluster_size=MAX_CLUSTER_SIZE)
        num_communities = len(set(c.cluster for c in clusters))
        print(f"Found {num_communities} communities")

        community_info = self._collect_community_info(nx_graph, clusters)
        self._generate_summaries(community_info)
        print(f"\n✅ {len(self.community_summaries)} community summaries generated")

    def _to_networkx(self) -> nx.Graph:
        nx_graph = nx.Graph()
        for node in self.graph.nodes.values():
            if isinstance(node, EntityNode):
                nx_graph.add_node(node.id)
        for relation in self.graph.relations.values():
            if relation.source_id in nx_graph and relation.target_id in nx_graph:
                nx_graph.add_edge(
                    relation.source_id,
                    relation.target_id,
                    relationship=relation.label,
                    description=relation.properties.get("relationship_description", ""),
                )
        return nx_graph

    def _collect_community_info(self, nx_graph, clusters) -> dict:
        community_mapping = {item.node: item.cluster for item in clusters}

        node_details = {}
        for node in self.graph.nodes.values():
            if not isinstance(node, EntityNode):
                continue
            node_details[node.id] = {
                "name":        node.name,
                "type":        node.label,
                "description": node.properties.get("entity_description", ""),
            }

        community_info = {}
        for item in clusters:
            cid, nid = item.cluster, item.node
            community_info.setdefault(cid, {"entities": [], "relationships": []})

            if nid in node_details:
                community_info[cid]["entities"].append(node_details[nid])

            for neighbor in nx_graph.neighbors(nid):
                if community_mapping.get(neighbor) == cid:
                    edge = nx_graph.get_edge_data(nid, neighbor)
                    rel  = edge.get("relationship", "RELATED") if edge else "RELATED"
                    desc = edge.get("description",  "")        if edge else ""

                    src_name = node_details.get(nid,      {}).get("name", nid)
                    tgt_name = node_details.get(neighbor, {}).get("name", neighbor)

                    entry = f"{src_name} --[{rel}]--> {tgt_name}"
                    if desc:
                        entry += f" ({desc})"
                    community_info[cid]["relationships"].append(entry)

        return community_info

    def _generate_summaries(self, community_info):
        import time
        for community_id, data in community_info.items():
            if not data["relationships"] and not data["entities"]:
                continue

            entities_text = "\n".join([
                f"- {e['name']} ({e['type']}): {e['description']}"
                for e in data["entities"] if e.get("name")
            ])

            relationships_text = "\n".join(sorted(set(data["relationships"])))

            prompt = f"""You are analysing a cluster of entities from news articles about
            AI copyright, governance, and intellectual property.

            Entities in this cluster:
            {entities_text}

            Relationships:
            {relationships_text}

            Write a concise briefing (3-5 sentences) that:
            1. Identifies the main organizations, people, legal cases, or topics in this cluster
            2. Explains how they are connected and why — including legal or regulatory context
            3. Highlights any disputes, lawsuits, policy positions, or tensions
            4. Notes anything particularly relevant for understanding AI copyright or governance

            Briefing:"""

            max_retries = 3
            for attempt in range(max_retries):
                try:
                    time.sleep(4)
                    # rotate key on each community summary call
                    current_llm = get_next_llm()
                    response = current_llm.complete(prompt)
                    self.community_summaries[community_id] = response.text
                    print(f"  Community {community_id}: {response.text[:100]}...")
                    break
                except Exception as e:
                    print(f"  Community {community_id} error (attempt {attempt+1}): {e}")
                    time.sleep(8)  # wait before rotating to next key

    def get_community_summaries(self) -> dict:
        return self.community_summaries

print("✅ GraphRAGStore defined")

✅ GraphRAGStore defined


### GraphRAG Query Engine

This code defines how users can **ask questions to the knowledge graph**. First, it checks each community summary and uses the LLM to find which communities contain relevant information. Then, it collects all relevant answers and sends them to another LLM to combine them into **one clear final answer**.

In simple terms:

```text
User Question
      ↓
Check all Community Summaries
      ↓
Find Relevant Answers
      ↓
Combine Relevant Information
      ↓
Generate Final Answer

In [17]:
class GraphRAGQueryEngine(CustomQueryEngine):
    """
    Queries all community summaries and synthesises a single answer.

    Step 1: For each community summary, ask EXTRACTION_LLM to answer the
            question based only on that summary. If not relevant, skip.
    Step 2: Aggregate all relevant partial answers using QUERY_LLM into
            one final, clear, non-redundant response.
    """

    graph_store: GraphRAGStore
    llm: LLM

    def custom_query(self, query_str: str) -> str:
        summaries = self.graph_store.get_community_summaries()

        if not summaries:
            return "No community summaries found. Run graph_store.build_communities() first."

        # Step 1: Get a partial answer from each community summary
        community_answers = [
            self._answer_from_community(summary, query_str)
            for summary in summaries.values()
        ]

        # Filter out empty/irrelevant responses
        relevant_answers = [a for a in community_answers if a.strip()]

        if not relevant_answers:
            return "I don't have enough information in the knowledge graph to answer that question."

        # Step 2: Aggregate into one final answer
        return self._aggregate(relevant_answers, query_str)

    def _answer_from_community(self, summary: str, query: str) -> str:
        """
        Ask EXTRACTION_LLM to answer the query from a single community summary.
        Returns empty string if the summary isn't relevant to the question.
        We use the cheaper model here since this runs once per community.
        """
        prompt = (
            f"Community summary:\n{summary}\n\n"
            f"Question: {query}\n\n"
            f"If this summary contains information relevant to the question, answer it. "
            f"If not relevant, reply exactly: 'No relevant information.'\n\n"
            f"Answer:"
        )
        response = EXTRACTION_LLM.complete(prompt)
        text = response.text.strip()

        # Filter out non-answers
        return "" if "no relevant information" in text.lower() else text

    def _aggregate(self, answers: List[str], query: str) -> str:
        """
        Synthesise all relevant partial answers into one final response.
        Uses QUERY_LLM (gpt-4o) for better reasoning quality on the final step.
        """
        combined = "\n\n---\n\n".join(answers)
        prompt = (
            f"You have received answers from multiple knowledge graph communities about this question:\n\n"
            f"Question: {query}\n\n"
            f"Community answers:\n{combined}\n\n"
            f"Synthesise these into a single, clear, well-structured final answer. "
            f"Remove redundancy, keep all important details, and ensure the answer "
            f"directly addresses the question.\n\n"
            f"Final Answer:"
        )
        return self.llm.complete(prompt).text

print("✅ GraphRAGQueryEngine defined")

✅ GraphRAGQueryEngine defined


In [18]:
# Load from CSV ───────────────────────────────────────────────────
df = pd.read_csv("ai_copyright_dataset.csv")
print(f"Number of articles: {len(df)}")
df.head()

Number of articles: 13


,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,NaN,success
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI is rapidly transforming the creative and in...,WIPO - World Intellectual Property Organization,NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,NaN,success
2,AI intellectual property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,NaN,success
3,AI intellectual property,Intellectual Property Rights and AI-Generated ...,The rise of generative AI is stress-testing ma...,"Medium · Adnan Masood, PhD.",NaN,https://medium.com/@adnanmasood/intellectual-p...,article,Member-only story\nIntellectual Property Right...,NaN,success
4,AI intellectual property,Artificial intelligence and intellectual prope...,An AI can generate a logo or visual close to a...,Cabinet Dreyfus,"Oct 6, 2025",https://www.dreyfus.fr/en/2025/10/06/artificia...,article,"A technological revolution, a legal vacuum\nAr...",NaN,success


### Creating Documents and Text Chunks

This code converts each row in the dataset into a LlamaIndex `Document`, including the article's text and metadata such as its title, source, and date.

Then, `SentenceSplitter` divides large documents into smaller text chunks called **nodes**. This makes the text easier for the LLM to process and extract entities and relationships from.

```text
Dataset
   ↓
Documents
   ↓
Split into smaller chunks (Nodes)
   ↓
Ready for GraphRAG extraction

In [19]:
from llama_index.core.node_parser import SentenceSplitter

documents = [
    Document(
        text=str(row["full_text"]),
        metadata={
            "title":  str(row.get("title",  "")),
            "source": str(row.get("source", "")),
            "date":   str(row.get("date",   "")),
        }
    )
    for _, row in df.iterrows()
]

splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=100)
nodes = splitter.get_nodes_from_documents(documents)

print(f"✅ Created {len(documents)} documents → split into {len(nodes)} nodes")

✅ Created 13 documents → split into 54 nodes


In [20]:
print(nodes[3].text)

Artificial Intelligence and Intellectual Property
Artificial intelligence (Al) is increasingly driving important developments in technology and business. It is being employed across a wide range of industries with impact on almost every aspect of the creation. The availability of large amounts of training data and advances in affordable high computing power are fueling Al's growth. Al intersects with intellectual property (IP) in a number of ways.
Featured
AI and IP Clearing House
AI is becoming a strategic capability for many governments across the globe. Strategies for the development of AI capacity and AI regulatory measures are being adopted with increasing frequency.
WIPO continuously collates and publishes the main government instruments of relevance to AI and IP with the aid of the Member States. Member States are invited to inform WIPO about any updates in their policies.
Eleventh session of the WIPO Conversation on AI and IP: Infrastructure for Rights Holders and Innovation
Co

### Building the Knowledge Graph

This code brings all the previous components together to **build the knowledge graph**.

First, `GraphRAGExtractor` is created with the extraction LLM and prompt. It processes each text chunk (`node`) and extracts entities and relationships.

Then, `GraphRAGStore` is created to store the extracted knowledge graph and later perform community detection.

Finally, `PropertyGraphIndex` processes all the nodes, uses the `kg_extractor` to extract entities and relationships, and stores them in the `graph_store`.

```text
Text Nodes
    ↓
GraphRAGExtractor
    ↓
Extract Entities + Relationships
    ↓
GraphRAGStore
    ↓
Knowledge Graph Created

In [21]:
# Instantiate the extractor with our ontology prompt
kg_extractor = GraphRAGExtractor(
    llm=EXTRACTION_LLM,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
)

# GraphRAGStore is both the graph database and the community detection engine
graph_store = GraphRAGStore()

print("Building knowledge graph... (this may take a few minutes)")

index = PropertyGraphIndex(
    nodes=nodes,
    kg_extractors=[kg_extractor],
    property_graph_store=graph_store,
    embed_kg_nodes=False,             # we don't need vector similarity search for this GraphRAG pipeline
    show_progress=True,
)

Building knowledge graph... (this may take a few minutes)


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

  Extraction error (attempt 1): 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash
Please retry in 38.733145488s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.6-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 38
}
]
  ✅ Extracted 14 entities, 9 relationships
  ✅ Extracted 10 entities, 4 relat

Applying transformations: 100%|██████████| 1/1 [30:58<00:00, 1858.04s/it]


### Inspecting Extracted Entities and Relationships

This code is used to **check and inspect the knowledge extraction results** for one text chunk.

It selects a sample node, prints its **title and original text**, and then runs the `GraphRAGExtractor` on a copy of that node. Using a copy ensures the original node is not changed.

Finally, it prints all the **entities and relationships** extracted by the LLM.

```text
Sample Article/Text
       ↓
Run GraphRAGExtractor
       ↓
Extract Entities
       ↓
Extract Relationships
       ↓
Print Results for Inspection

In [22]:
# Print out example of an article and the extracted entities & relationships
import copy

# Change this index to inspect a different article
sample = nodes[3]

print("=== Title ===")
print(sample.metadata.get("title", "untitled"))
print()
print("=== Raw text ===")
print(sample.text)
print()

# Re-run extraction on a copy so we don't mutate the original
extracted = await kg_extractor._aextract(copy.deepcopy(sample))

print("=== Extracted entities ===")
for node in extracted.metadata.get(KG_NODES_KEY, []):
    print(f"  [{node.label}] {node.name}")
    print(f"    {node.properties.get('entity_description', '')}")

print()
print("=== Extracted relationships ===")
id_to_name = {n.id: n.name for n in extracted.metadata.get(KG_NODES_KEY, [])}
for rel in extracted.metadata.get(KG_RELATIONS_KEY, []):
    src = id_to_name.get(rel.source_id, rel.source_id)
    tgt = id_to_name.get(rel.target_id, rel.target_id)
    print(f"  {src} --[{rel.label}]--> {tgt}")

=== Title ===
Artificial Intelligence and Intellectual Property

=== Raw text ===
Artificial Intelligence and Intellectual Property
Artificial intelligence (Al) is increasingly driving important developments in technology and business. It is being employed across a wide range of industries with impact on almost every aspect of the creation. The availability of large amounts of training data and advances in affordable high computing power are fueling Al's growth. Al intersects with intellectual property (IP) in a number of ways.
Featured
AI and IP Clearing House
AI is becoming a strategic capability for many governments across the globe. Strategies for the development of AI capacity and AI regulatory measures are being adopted with increasing frequency.
WIPO continuously collates and publishes the main government instruments of relevance to AI and IP with the aid of the Member States. Member States are invited to inform WIPO about any updates in their policies.
Eleventh session of the W

### Displaying Unique Entities

This code organizes and displays all the entities stored in the knowledge graph based on their **entity type**.

For example, it groups entities like this:

```text
ORGANIZATION
  OpenAI
  Google

PERSON
  Sam Altman

AI_SYSTEM
  GPT-4
  Gemini

In [23]:
# ── Print out unique entities extracted ───────────────────────────────────────────────────
from collections import defaultdict

by_type = defaultdict(list)
for node in graph_store.graph.nodes.values():
    if isinstance(node, EntityNode):
        by_type[node.label].append(node.name)

for entity_type, names in sorted(by_type.items()):
    print(f"\n{entity_type} ({len(names)})")
    for name in sorted(names):
        print(f"  {name}")


AI_SYSTEM (26)
  AI COMPANIONS
  ANALYTICAL ENGINE
  BERT
  CHATGPT
  CLIP
  COPILOT
  ChatGPT
  Creative Adversarial Networks
  DABUS
  DALL-E
  DALL-E 2
  DALL·E 2
  GAI
  GENAI
  GENERATIVE ARTIFICIAL INTELLIGENCE
  GPAI MODELS
  GPT SERIES
  GPT-3
  Generative AI
  Generative AI Models
  LAION-5B
  Large Language Models
  MIDJOURNEY
  STABLE DIFFUSION
  Stable Diffusion
  THE PAINTING FOOL

CONCEPT (61)
  AESTHETIC NEUTRALITY
  AESTHETIC NEUTRALITY DOCTRINE
  AI INVENTIONS
  AI TRAINING
  AI-GENERATED WORKS
  AI-Generated Art
  ARTIFICIAL INTELLIGENCE
  ARTISTIC TURING TEST
  ARTISTS
  CHOKEPOINT CAPITALISM
  COMMONPOOL
  COPYRIGHT
  COPYRIGHT POISONING
  COPYRIGHTED CONTENT
  COPYRIGHTED MATERIAL
  COPYRIGHTED WORKS
  CREATIVITY
  Copyrighted Material
  Copyrighted Materials
  DEEPFAKES
  DIFFUSION MODELS
  DIGITAL REPLICAS
  FAIR USE
  FAIR USE DOCTRINE
  FOUNDATION MODELS
  Fair Use
  GENERATIVE ADVERSARIAL NETWORKS
  GENERATIVE AI
  GHOST WORK
  GHOST WORKERS
  GOOGLE BOOKS PR

### Inspecting a Specific Entity

This code is used to **inspect one specific entity** in the knowledge graph, such as `STABLE DIFFUSION`.

It searches for the entity and, if found, displays its **name, type, source, title, original article text, and all relationships connected to it**. If the entity is not found, it prints a list of available entities.

```text
Select Entity
     ↓
Find Entity in Graph
     ↓
Show Entity Details
     ↓
Find Original Article
     ↓
Display Connected Relationships

In [24]:
# ── Inspect a specific untyped entity ─────────────────────────────────────────
inspect_name = "STABLE DIFFUSION"  # change to any name from your entity list

# Find the node (filter to EntityNode only — graph also contains ChunkNodes)
node = next(
    (n for n in graph_store.graph.nodes.values()
     if isinstance(n, EntityNode) and n.name == inspect_name),
    None
)

if node is None:
    print(f"❌ Entity '{inspect_name}' not found in graph.")
    print("Available entities:")
    for n in graph_store.graph.nodes.values():
        if isinstance(n, EntityNode):
            print(f"  {n.name}")
else:
    print(f"Node:   {node.name!r}  label={node.label!r}")
    print(f"Source: {node.properties.get('source', 'N/A')}")
    print(f"Title:  {node.properties.get('title', 'N/A')}")
    print()

    # Look up the original article text by matching the title
    title = node.properties.get("title")
    article = next((n for n in nodes if n.metadata.get("title") == title), None)
    if article:
        print("=== Article text ===")
        print(article.text)
    print()

    # Find all relations involving this entity
    relations = [
        r for r in graph_store.graph.relations.values()
        if r.source_id == node.id or r.target_id == node.id
    ]
    node_index = {
        n.id: n.name for n in graph_store.graph.nodes.values()
        if isinstance(n, EntityNode)
    }
    print(f"Relations ({len(relations)}):")
    for r in relations:
        src = node_index.get(r.source_id, r.source_id)
        tgt = node_index.get(r.target_id, r.target_id)
        print(f"  {src} --[{r.label}]--> {tgt}")

Node:   'STABLE DIFFUSION'  label='AI_SYSTEM'
Source: Yale Law Journal
Title:  ARTificial: Why Copyright Is Not the Right Policy Tool to ...

=== Article text ===
ARTificial: Why Copyright Is Not the Right Policy Tool to Deal with Generative AI
abstract. The rapid advancement and widespread application of Generative Artificial Intelligence (GAI) raise complex issues regarding authorship, originality, and the ethical use of copyrighted materials for AI training.
As attempts to regulate AI proliferate, this Essay proposes a taxonomy of reasons, from the perspective of creatives and society alike, that explain why copyright law is ill-equipped to handle the nuances of AI-generated content.
Originally designed to incentivize creativity, copyright doctrine has been expanded in scope to cover new technological mediums. This expansion has proven to increase the complexity and uncertainty of copyright doctrine’s application—ironically leading to the stifling of innovation. In this Essay, I war

### Saving the Knowledge Graph

This code saves the `graph_store` object to a file called `graph_store.pkl` using Python's `pickle` library. This allows you to load and reuse the knowledge graph later without rebuilding it from scratch.

It also counts the total number of extracted entities and prints a confirmation message showing how many entities and communities were saved.

```text
Knowledge Graph
      ↓
Save using Pickle
      ↓
graph_store.pkl
      ↓
Load and reuse later

In [26]:
import pickle

with open("graph_store.pkl", "wb") as f:
    pickle.dump(graph_store, f)

entity_count = sum(1 for n in graph_store.graph.nodes.values() if isinstance(n, EntityNode))
print(f"✅ Saved graph with {entity_count} entities and 69 communities to disk")

✅ Saved graph with 235 entities and 69 communities to disk


```text
Knowledge Graph
      ↓
Leiden Community Detection
      ↓
Collect Entities + Relationships
      ↓
Send Each Community to Gemini Directly
      ↓
Generate and Store Summaries

In [37]:
import google.generativeai as genai
import time
import warnings
from dotenv import load_dotenv

load_dotenv(override=True)

# Configure directly with the API
fresh_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=fresh_key)
direct_model = genai.GenerativeModel("gemini-3.5-flash-lite")

print("✅ Direct Gemini API configured with gemini-3.5-flash-lite")

# Generate summaries directly
graph_store.community_summaries = {}
community_info = graph_store._collect_community_info(
    graph_store._to_networkx(),
    __import__('graspologic').partition.hierarchical_leiden(
        graph_store._to_networkx(), 
        max_cluster_size=MAX_CLUSTER_SIZE
    )
)

print(f"Generating summaries for {len(community_info)} communities...")

for community_id, data in community_info.items():
    if not data["relationships"] and not data["entities"]:
        continue
    
    entities_text = "\n".join([
        f"- {e['name']} ({e['type']}): {e['description']}"
        for e in data["entities"] if e.get("name")
    ])
    relationships_text = "\n".join(sorted(set(data["relationships"])))
    
    prompt = f"""Analyse this cluster of AI copyright/governance entities.
Entities: {entities_text}
Relationships: {relationships_text}
Write a concise 3-5 sentence briefing covering main entities, connections, disputes, and AI copyright relevance.
Briefing:"""

    for attempt in range(3):
        try:
            time.sleep(4)
            response = direct_model.generate_content(prompt)
            graph_store.community_summaries[community_id] = response.text
            print(f"  ✅ Community {community_id}: {response.text[:80]}...")
            break
        except Exception as e:
            print(f"  Community {community_id} error (attempt {attempt+1}): {e}")
            time.sleep(8)

print(f"\n✅ {len(graph_store.community_summaries)} community summaries generated")

✅ Direct Gemini API configured with gemini-3.5-flash-lite
Generating summaries for 71 communities...
  ✅ Community 0: This cluster of entities centers on the intersection of generative AI, copyright...
  ✅ Community 1: This cluster of entities centers on the intense legal, economic, and ethical dis...
  ✅ Community 2: This briefing centers on the proposed **No AI Fraud Act** and its sponsor, US Re...
  ✅ Community 3: This cluster of entities centers on the European Union’s robust legal framework ...
  ✅ Community 4: This cluster examines the intersection of generative artificial intelligence (Ge...
  ✅ Community 5: This entity cluster centers on *Andersen v. Stability AI Ltd.*, a major class-ac...
  ✅ Community 6: This cluster centers on the landmark **DABUS LITIGATION**, spearheaded by creato...
  ✅ Community 7: This briefing examines the relationship between intellectual property law firm N...
  ✅ Community 8: This cluster centers on the **U.S. Copyright Office** and its comprehensiv

✅ Visualization saved to ai_copyright_graph.html
   Nodes: 235
   Edges: 264
   Communities: 71


In [42]:
import json
import pathlib


def export_graph_data(graph_store: GraphRAGStore, output_file: str = "graph_data.json"):
    """
    Export the knowledge graph to a JSON file compatible with the D3.js template.
    Run this once after graph_store.build_communities().

    Saves to disk so you can re-run the visualization without reprocessing
    the full pipeline — which is expensive.
    """

    nx_graph = graph_store._to_networkx()

    # ── Build node metadata lookup ─────────────────────────────────────────
    node_meta = {}
    for node in graph_store.graph.nodes.values():
        if isinstance(node, EntityNode):
            node_meta[node.id] = {
                "id":          node.id,
                "label":       node.name,
                "type":        node.label if node.label else "OTHER",
                "description": node.properties.get("entity_description", ""),
            }

    # ── Nodes — only those present in the NetworkX graph ──────────────────
    nodes_data = [
        node_meta[n] for n in nx_graph.nodes() if n in node_meta
    ]

    # ── Links ──────────────────────────────────────────────────────────────
    links_data = []
    for source, target, data in nx_graph.edges(data=True):
        links_data.append({
            "source":      source,
            "target":      target,
            "label":       data.get("relationship", ""),
            "description": data.get("description",  ""),
        })

    # ── Assemble and write ─────────────────────────────────────────────────
    graph_data = {
        "nodes":       nodes_data,
        "links":       links_data,
        "communities": len(graph_store.get_community_summaries()),
    }

    pathlib.Path(output_file).write_text(json.dumps(graph_data, indent=2))

    print(f"✅ Graph data exported to '{output_file}'")
    print(f"   Nodes:       {len(nodes_data)}")
    print(f"   Edges:       {len(links_data)}")
    print(f"   Communities: {graph_data['communities']}")

    return graph_data


def visualize_graph(
    graph_store: GraphRAGStore = None,
    graph_data_file: str = "graph_data.json",
    template_file: str = "graph_template.html",
    output_file: str = GRAPH_OUTPUT_FILE,
):
    """
    Generate a D3.js interactive knowledge graph visualization.

    Can be called in two ways:

    1. Pass graph_store directly — exports fresh data then builds the HTML:
         visualize_graph(graph_store=graph_store)

    2. Load from a previously exported JSON file — no pipeline re-run needed:
         visualize_graph()  # loads graph_data.json automatically

    Features:
    - Color-coded nodes by entity type (from ontology)
    - Node size scales with degree (more connections = bigger)
    - Click a node to highlight its neighbourhood, dim everything else
    - Sidebar legend — click any entity type to toggle its visibility
    - Search bar to find and highlight any node by name
    - Sliders for link distance, charge strength, and edge label threshold
    - Hover tooltips with entity name, type, description, and connection count
    - Stats header showing total nodes, edges, and communities
    """

    # ── Get graph data ─────────────────────────────────────────────────────
    if graph_store is not None:
        # Export fresh data from the graph store and save to disk
        graph_data = export_graph_data(graph_store, graph_data_file)
    else:
        # Load from a previously saved JSON file
        data_path = pathlib.Path(graph_data_file)
        if not data_path.exists():
            print(f"⚠️  '{graph_data_file}' not found.")
            print(f"   Run export_graph_data(graph_store) first, or pass graph_store directly.")
            return
        graph_data = json.loads(data_path.read_text())

        print(f"✅ Loaded graph data from '{graph_data_file}'")
        print(f"   Nodes: {len(graph_data['nodes'])}  |  "
              f"Edges: {len(graph_data['links'])}  |  "
              f"Communities: {graph_data.get('communities', '—')}")

    # ── Load HTML template ─────────────────────────────────────────────────
    template_path = pathlib.Path(template_file)
    if not template_path.exists():
        print(f"⚠️  '{template_file}' not found.")
        print(f"   Make sure graph_template.html is in the same folder as this notebook.")
        return

    # ── Inject data into template and write output ─────────────────────────
    html = template_path.read_text()
    html = html.replace("GRAPH_DATA_PLACEHOLDER", json.dumps(graph_data))
    pathlib.Path(output_file).write_text(html)

    print(f"\n✅ Visualization saved to '{output_file}'")
    print(f"   Open it in your browser to explore.")


# Export data and build visualization from the graph store
# visualize_graph(graph_store=graph_store)

In [43]:
import json
import pathlib

# Read the already-saved graph data
graph_data = json.loads(pathlib.Path("graph_data.json").read_text(encoding="utf-8"))

# Read the template with explicit UTF-8 encoding
html = pathlib.Path("graph_template.html").read_text(encoding="utf-8")

# Inject the data
html = html.replace("GRAPH_DATA_PLACEHOLDER", json.dumps(graph_data))

# Write output with UTF-8 encoding
pathlib.Path("ai_copyright_graph.html").write_text(html, encoding="utf-8")

print(f"✅ Visualization saved to ai_copyright_graph.html")
print(f"   Nodes: {len(graph_data['nodes'])}")
print(f"   Edges: {len(graph_data['links'])}")
print(f"   Communities: {graph_data['communities']}")

✅ Visualization saved to ai_copyright_graph.html
   Nodes: 235
   Edges: 264
   Communities: 71


In [44]:
query_engine = GraphRAGQueryEngine(
    graph_store=graph_store,
    llm=QUERY_LLM,
)

print("✅ Query engine ready")

✅ Query engine ready


In [56]:
import google.generativeai as genai
import time
from dotenv import load_dotenv

load_dotenv(override=True)
genai.configure(api_key=os.getenv("GEMINI_API_KEY_4"))
direct_query_model = genai.GenerativeModel("gemini-3.5-flash-lite")

def call_with_retry(model, prompt, max_retries=5, base_wait=15):
    for attempt in range(max_retries):
        try:
            return model.generate_content(prompt)
        except Exception as e:
            if "429" in str(e):
                print(f"  Rate limited, waiting {base_wait}s (attempt {attempt+1}/{max_retries})")
                time.sleep(base_wait)
            else:
                raise
    raise RuntimeError("Max retries exceeded on rate limit")

def query_graph(question: str) -> str:
    summaries = graph_store.get_community_summaries()
    if not summaries:
        return "No community summaries found. Run graph_store.build_communities() first."
    relevant_answers = []
    for community_id, summary in summaries.items():
        prompt = (
            f"Community summary:\n{summary}\n\n"
            f"Question: {question}\n\n"
            f"If relevant, answer it. If not, reply: 'No relevant information.'\nAnswer:"
        )
        try:
            time.sleep(4.5)  # ~13 requests/min, safely under the 15 RPM cap
            response = call_with_retry(direct_query_model, prompt)
            text = response.text.strip()
            if "no relevant information" not in text.lower():
                relevant_answers.append(text)
        except Exception as e:
            print(f"  Community {community_id} error: {e}")
    if not relevant_answers:
        return "Not enough information in the knowledge graph."
    combined = "\n\n---\n\n".join(relevant_answers)
    final = direct_query_model.generate_content(
        f"Question: {question}\n\nCommunity answers:\n{combined}\n\nSynthesise into one clear answer.\nFinal Answer:"
    )
    return final.text

print("✅ query_graph() ready")

✅ query_graph() ready


In [60]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv(override=True) # sanity check, don't print full key

genai.configure(api_key=os.getenv("GEMINI_API_KEY_4"))
model = genai.GenerativeModel("gemini-3.5-flash-lite")

resp = model.generate_content("Say hello")
print(resp.text)

Hello! How can I help you today?


In [58]:
response = query_graph(
    "What are the main legal disputes involving AI companies and copyright holders?"
)
print(response)

Based on the provided community summaries, the main legal disputes involving AI companies and copyright holders center on two primary areas: **AI training data ingestion** and **AI-generated outputs/authorship**. 

### 1. AI Training Data Ingestion and Copyright Infringement
The most prominent legal battles involve copyright holders (such as visual artists, authors, major media publishers like *The New York Times*, and labor unions like SAG-AFTRA) suing tech giants and AI developers (such as OpenAI, Microsoft, Stability AI, Midjourney, and DeviantArt). 
* **The Core Issue:** AI companies scrape massive amounts of copyrighted text, images, and creative works from the internet to train foundational models and large language models (LLMs) without permission or compensation.
* **The Legal Debate:** Plaintiffs argue this constitutes large-scale copyright infringement and economic harm. Conversely, AI companies defend the practice under the **Fair Use Doctrine**, arguing that training machin

In [59]:


response = query_graph(
    "How are different governments (e.g. EU, US, and UK) approaching AI governance?"
)
print(response)

Based on the provided community summaries, governments like the EU, US, and international bodies are approaching AI governance primarily through intellectual property frameworks, transparency mandates, and legal rulings. Their approaches can be broken down as follows:

*   **United States:** The U.S. approaches AI governance through a combination of legislative proposals, agency guidance, and judicial rulings. 
    *   *Copyright & Authorship:* Courts (such as the DC Circuit in *Thaler v. Perlmutter*), the U.S. Copyright Office, and the USPTO strictly condition copyright eligibility and patent filings on significant **human creation and contribution**, denying independent legal personhood or authorship to AI.
    *   *Legislation & Regulation:* Lawmakers have introduced frameworks like the *Generative AI Copyright Disclosure Act of 2024* (via Congress and the House/Senate Judiciary Committees) to address training data transparency, while established legal doctrines like *Fair Use* (e.g